# Week 3: Diffusion Policy on a Single LIBERO Kitchen Task

**Vizuara Robot Learning Bootcamp**

In Weeks 1-2 you trained Diffusion Policy on PushT (2D) and ALOHA (bimanual). Now we step into **3D kitchen manipulation** with the LIBERO benchmark.

This notebook is deliberately focused: **one task, one policy, one goal — make it work.**

You will:
1. Set up the LIBERO simulation environment (MuJoCo + robosuite)
2. Explore a single kitchen task — visualize dual-camera demos, robot state, and actions
3. Train Diffusion Policy on ~50 demonstrations of that task
4. Evaluate the trained policy in simulation and render a video
5. Understand what changed from PushT → LIBERO (action space, cameras, state)

> **Hardware**: Google Colab T4 GPU (free tier works). Training takes ~30 min for 20K steps.
>
> **Prerequisites**: Week 1 (PushT Diffusion Policy) and Week 2 (ALOHA). You should understand DDPM, 1D U-Net, FiLM conditioning, and `crop_shape`.

---

## Part 0: Setup & Installation

LIBERO runs on MuJoCo via robosuite. On Colab (no display), we need EGL for headless GPU rendering.

In [ ]:
# Step 1: Install EGL for headless MuJoCo rendering (MUST be first)
!apt-get install -qq -y libegl1-mesa-dev libgl1-mesa-dev libgles2-mesa-dev > /dev/null 2>&1

# Step 2: Install LeRobot with LIBERO support
!pip install -q "lerobot[libero]" mediapy 2>&1 | tail -5

# Step 3: Set MuJoCo rendering backend BEFORE any import
import os
os.environ["MUJOCO_GL"] = "egl"

# Verify
import lerobot
print(f"LeRobot version: {lerobot.__version__}")
print(f"MUJOCO_GL: {os.environ.get('MUJOCO_GL')}")

try:
    from libero.libero import benchmark
    print(f"LIBERO benchmark: OK")
    print(f"Available suites: {list(benchmark.get_benchmark_dict().keys())}")
except ImportError:
    print("ERROR: LIBERO not installed. Run: pip install lerobot[libero]")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from IPython.display import HTML, display
import mediapy as media
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Part 1: Meet LIBERO — The Kitchen Manipulation Benchmark

### What is LIBERO?

[LIBERO](https://libero-project.github.io/) (Liu et al., NeurIPS 2023) is a benchmark with 130+ manipulation tasks in simulated kitchen environments. A Franka Panda robot performs tasks like picking, placing, opening drawers, and pouring — all described by natural language instructions.

### The Robot

| | Details |
|---|---|
| **Robot** | Franka Emika Panda (7-DOF) |
| **Cameras** | 2: overhead (agentview) + wrist (eye-in-hand), both 256×256 |
| **State** | 8D: end-effector position (3) + orientation as axis-angle (3) + gripper (2) |
| **Action** | 7D relative: [dx, dy, dz, rx, ry, rz, gripper] ∈ [-1, 1] |
| **Control** | 10 Hz |

### Comparison to Previous Weeks

| | PushT (Week 1) | ALOHA (Week 2) | **LIBERO (This Week)** |
|---|---|---|---|
| Action dim | 2 (x, y) | 14 (7×2 arms) | **7 (Cartesian)** |
| Image | 96×96 ×1 cam | 480×640 ×1 cam | **256×256 ×2 cams** |
| State dim | 2 | 14 | **8** |
| Environment | 2D pushing | Real bimanual | **3D kitchen sim** |
| Demos | 205 | 50 | **~50** |

---
## Part 2: Explore the Dataset

We'll use the LIBERO-Spatial suite. Let's find a good kitchen task and explore its demonstrations.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

# Load metadata (lightweight — no video download yet)
meta = LeRobotDatasetMetadata("HuggingFaceVLA/libero")

print(f"Dataset: HuggingFaceVLA/libero")
print(f"  FPS: {meta.fps}")
print(f"  Total episodes: {meta.total_episodes}")
print(f"  Total frames: {meta.total_frames}")

# List all tasks
print(f"\n=== All {len(meta.tasks)} Tasks ===")
for idx, row in meta.tasks.iterrows():
    print(f"  Task {row['task_index']:2d}: {idx}")

In [ ]:
# Build task-to-episode mapping from metadata
task_episodes = defaultdict(list)
task_descriptions = {}

for task_name, row in meta.tasks.iterrows():
    task_descriptions[row['task_index']] = str(task_name)

for ep_idx in range(len(meta.episodes)):
    ep = meta.episodes[ep_idx]
    ep_tasks = ep.get('tasks', [])
    if isinstance(ep_tasks, list) and len(ep_tasks) > 0:
        task_str = ep_tasks[0]
        if task_str in meta.tasks.index:
            tidx = int(meta.tasks.loc[task_str, 'task_index'])
            task_episodes[tidx].append(ep_idx)

# Show tasks with episode counts
print("Tasks with available demos:")
for tidx in sorted(task_episodes.keys()):
    desc = task_descriptions.get(tidx, "?")
    print(f"  Task {tidx:2d} ({len(task_episodes[tidx]):2d} eps): {desc}")

In [ ]:
# ============================================================
# CHOOSE YOUR TASK HERE
# Pick a LIBERO-Spatial task (task indices 0-9 are spatial)
# These are kitchen tasks where the same objects are in different positions.
# ============================================================

TARGET_TASK_IDX = 0  # Change this to try a different task!

target_episodes = task_episodes[TARGET_TASK_IDX]
task_name = task_descriptions[TARGET_TASK_IDX]

print(f"Selected task: {task_name}")
print(f"Available demonstrations: {len(target_episodes)}")
print(f"Episode indices: {target_episodes}")

In [ ]:
# Load ONE episode to explore the data format
ds_explore = LeRobotDataset("HuggingFaceVLA/libero", episodes=[target_episodes[0]])
print(f"Episode {target_episodes[0]}: {len(ds_explore)} frames ({len(ds_explore)/meta.fps:.1f}s)")

sample = ds_explore[0]
print(f"\n=== Sample Keys ===")
for key, val in sorted(sample.items()):
    if isinstance(val, torch.Tensor):
        print(f"  {key:40s} {str(val.shape):15s} {val.dtype}")
    else:
        print(f"  {key:40s} = {val}")

In [ ]:
# Visualize the dual-camera setup across an episode
frame_indices = np.linspace(0, len(ds_explore) - 1, 6, dtype=int)

fig, axes = plt.subplots(2, 6, figsize=(24, 8))

for col, idx in enumerate(frame_indices):
    s = ds_explore[idx]
    t_sec = idx / meta.fps

    # Agentview (overhead camera)
    for img_key in ['observation.images.image', 'observation.image']:
        if img_key in s:
            img = s[img_key]
            if img.dim() == 4: img = img[-1]
            img_np = img.permute(1, 2, 0).numpy()
            if img_np.max() > 1.0: img_np = img_np / 255.0
            axes[0, col].imshow(np.clip(img_np, 0, 1))
            break
    axes[0, col].set_title(f't={t_sec:.1f}s', fontsize=10)
    axes[0, col].axis('off')
    if col == 0:
        axes[0, col].set_ylabel('Agentview\n(overhead)', fontsize=12, fontweight='bold')

    # Wrist camera
    for img_key2 in ['observation.images.image2', 'observation.image2']:
        if img_key2 in s:
            img2 = s[img_key2]
            if img2.dim() == 4: img2 = img2[-1]
            img2_np = img2.permute(1, 2, 0).numpy()
            if img2_np.max() > 1.0: img2_np = img2_np / 255.0
            axes[1, col].imshow(np.clip(img2_np, 0, 1))
            break
    axes[1, col].axis('off')
    if col == 0:
        axes[1, col].set_ylabel('Wrist\n(eye-in-hand)', fontsize=12, fontweight='bold')

plt.suptitle(f'Demo Episode: "{task_name}"\nDual cameras capture global context + fine detail',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the robot state and actions over one episode
states, actions = [], []
for i in range(len(ds_explore)):
    s = ds_explore[i]
    if 'observation.state' in s:
        states.append(s['observation.state'].numpy())
    if 'action' in s:
        actions.append(s['action'].numpy())

states = np.array(states)
actions = np.array(actions)
t = np.arange(len(states)) / meta.fps

state_names = ['EEF x', 'EEF y', 'EEF z', 'Rot 1', 'Rot 2', 'Rot 3', 'Grip L', 'Grip R']
action_names = ['dx', 'dy', 'dz', 'rx', 'ry', 'rz', 'gripper']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# EEF position over time
colors = ['#D97757', '#5B8CB8', '#7DA488']
for i in range(3):
    axes[0, 0].plot(t, states[:, i], label=state_names[i], color=colors[i], linewidth=2)
axes[0, 0].set_ylabel('Position (m)')
axes[0, 0].set_title('End-Effector Position', fontweight='bold')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# Gripper state
axes[0, 1].plot(t, states[:, 6], label='Grip L', color='#9B7EC8', linewidth=2)
axes[0, 1].plot(t, states[:, 7], label='Grip R', color='#C4956A', linewidth=2)
axes[0, 1].set_ylabel('Gripper position')
axes[0, 1].set_title('Gripper State', fontweight='bold')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

# Translation actions
for i in range(3):
    axes[1, 0].plot(t[:len(actions)], actions[:, i], label=action_names[i],
                    color=colors[i], alpha=0.8)
axes[1, 0].set_xlabel('Time (s)'); axes[1, 0].set_ylabel('Action value')
axes[1, 0].set_title('Translation Actions (dx, dy, dz)', fontweight='bold')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

# Gripper action
axes[1, 1].plot(t[:len(actions)], actions[:, 6], color='#9B7EC8', linewidth=2)
axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].annotate('Open', xy=(0, 0.7), fontsize=11, color='green')
axes[1, 1].annotate('Close', xy=(0, -0.7), fontsize=11, color='red')
axes[1, 1].set_xlabel('Time (s)'); axes[1, 1].set_ylabel('Gripper command')
axes[1, 1].set_title('Gripper Action', fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.suptitle(f'Demo Trajectory: "{task_name}"', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"State shape: {states.shape} — 8D: [EEF_pos(3), EEF_rot(3), gripper(2)]")
print(f"Action shape: {actions.shape} — 7D: [dx, dy, dz, rx, ry, rz, gripper]")

In [ ]:
# 3D trajectory visualization
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Color by time
scatter = ax.scatter(states[:, 0], states[:, 1], states[:, 2],
                     c=t, cmap='viridis', s=15, alpha=0.8)
ax.scatter(*states[0, :3], c='green', s=100, marker='o', label='Start', zorder=5)
ax.scatter(*states[-1, :3], c='red', s=100, marker='X', label='End', zorder=5)
ax.plot(states[:, 0], states[:, 1], states[:, 2], alpha=0.3, color='gray')

ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.set_title(f'End-Effector 3D Path\n"{task_name}"', fontweight='bold')
ax.legend(fontsize=10)
plt.colorbar(scatter, label='Time (s)', shrink=0.6)
plt.tight_layout()
plt.show()

---
## Part 3: Configure and Train Diffusion Policy

Now we train Diffusion Policy on ALL demonstrations for our chosen task.

### Key Differences from PushT (Week 1)

| | PushT | LIBERO |
|---|---|---|
| **Images** | 1 camera, 96×96 | **2 cameras**, 256×256 |
| **crop_shape** | (84, 84) default | **(224, 224)** — must change! |
| **State** | 2D position | 8D: EEF pos + rot + gripper |
| **Action** | 2D (x, y) | 7D Cartesian + gripper |
| **Demos** | 205 episodes | ~50 episodes |

**The `crop_shape` lesson from Week 2 applies here too**: the default (84, 84) would destroy 256×256 images. We set it to (224, 224).

In [ ]:
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.configs.types import FeatureType
from lerobot.datasets.utils import dataset_to_policy_features

# Build input/output feature specs from dataset metadata
features = dataset_to_policy_features(meta.features)
output_features = {k: ft for k, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {k: ft for k, ft in features.items() if k not in output_features}

print("=== Input Features ===")
for k, ft in input_features.items():
    print(f"  {k}: type={ft.type}, shape={ft.shape}")
print(f"\n=== Output Features ===")
for k, ft in output_features.items():
    print(f"  {k}: type={ft.type}, shape={ft.shape}")

# Detect image size for crop_shape
image_features = {k: ft for k, ft in input_features.items() if ft.type is FeatureType.VISUAL}
first_img_shape = next(iter(image_features.values())).shape
img_h, img_w = first_img_shape[1], first_img_shape[2]
crop_h = int(img_h * 0.875) // 8 * 8
crop_w = int(img_w * 0.875) // 8 * 8
print(f"\nImage size: {img_h}×{img_w}, crop_shape: ({crop_h}, {crop_w})")
print(f"Number of cameras: {len(image_features)}")

In [ ]:
# Helper: convert delta indices to timestamps
def make_delta_timestamps(delta_indices, fps):
    if delta_indices is None:
        return [0.0]
    return [i / fps for i in delta_indices]

# Configure Diffusion Policy
diff_cfg = DiffusionConfig(
    input_features=input_features,
    output_features=output_features,
    crop_shape=(crop_h, crop_w),    # NOT (84,84) — would destroy 256×256 images!
    crop_is_random=True,
    n_obs_steps=2,                  # Observe at t-1 and t
    horizon=16,                     # Predict 16-step trajectory
    n_action_steps=8,               # Execute first 8 steps
    num_train_timesteps=100,        # 100 diffusion steps
)

print("=== Diffusion Policy Config ===")
print(f"  crop_shape        = {diff_cfg.crop_shape}")
print(f"  n_obs_steps       = {diff_cfg.n_obs_steps}")
print(f"  horizon           = {diff_cfg.horizon} (predict 16 steps)")
print(f"  n_action_steps    = {diff_cfg.n_action_steps} (execute 8 steps)")
print(f"  num_train_ts      = {diff_cfg.num_train_timesteps}")

# Build delta_timestamps
delta_timestamps = {
    "action": make_delta_timestamps(diff_cfg.action_delta_indices, meta.fps),
}
delta_timestamps |= {
    k: make_delta_timestamps(diff_cfg.observation_delta_indices, meta.fps)
    for k in diff_cfg.image_features
}
delta_timestamps |= {
    k: make_delta_timestamps(diff_cfg.observation_delta_indices, meta.fps)
    for k in diff_cfg.state_features
}

print(f"\n=== delta_timestamps ===")
for k, v in delta_timestamps.items():
    print(f"  {k}: {len(v)} steps — {v[:4]}{'...' if len(v) > 4 else ''}")

In [ ]:
# Load ALL demonstrations for our single task
print(f"Loading {len(target_episodes)} episodes for: \"{task_name}\"")
print(f"Episode indices: {target_episodes}")
print("(This downloads video data — may take a few minutes on first run)\n")

dataset = LeRobotDataset(
    "HuggingFaceVLA/libero",
    episodes=target_episodes,
    delta_timestamps=delta_timestamps,
)

print(f"Dataset size: {len(dataset)} training samples")
print(f"From {len(target_episodes)} episodes of a single task")

# Verify sample shapes
sample = dataset[0]
print(f"\n=== Training Sample Shapes ===")
for key, val in sorted(sample.items()):
    if isinstance(val, torch.Tensor):
        print(f"  {key:40s} {str(val.shape)}")

In [ ]:
# Instantiate the policy
policy = DiffusionPolicy(diff_cfg)
policy.train()
policy.to(device)

preprocessor, postprocessor = make_pre_post_processors(
    diff_cfg, dataset_stats=meta.stats
)

n_params = sum(p.numel() for p in policy.parameters())
print(f"Diffusion Policy: {n_params:,} parameters ({n_params/1e6:.1f}M)")

# Compare to PushT (Week 1)
print(f"\nFor reference:")
print(f"  PushT policy:  ~26M params (1 camera, 2D action)")
print(f"  LIBERO policy: ~{n_params/1e6:.0f}M params (2 cameras, 7D action)")
print(f"  More cameras and action dimensions = more parameters")

In [ ]:
# ============================================================
# TRAINING
# ============================================================

TRAINING_STEPS = 20000   # Increase to 50K+ for better results
BATCH_SIZE = 8           # Reduce to 4 if you hit OOM
LR = 1e-4
LOG_FREQ = 500

optimizer = torch.optim.Adam(
    policy.parameters(), lr=LR,
    betas=(0.95, 0.999), eps=1e-8, weight_decay=1e-6,
)

dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)

print(f"Training Diffusion Policy on: \"{task_name}\"")
print(f"  Steps: {TRAINING_STEPS}, Batch: {BATCH_SIZE}, LR: {LR}")
print(f"  Dataset: {len(dataset)} samples from {len(target_episodes)} demos")
print(f"  Estimated time: ~{TRAINING_STEPS * 0.1 / 60:.0f} min on T4\n")

losses = []
step = 0
done = False
start_time = time.time()

while not done:
    for batch in dataloader:
        batch = preprocessor(batch)
        loss, _ = policy.forward(batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())

        if step % LOG_FREQ == 0:
            elapsed = time.time() - start_time
            avg = np.mean(losses[-LOG_FREQ:]) if len(losses) >= LOG_FREQ else np.mean(losses)
            speed = (step + 1) / elapsed
            eta = (TRAINING_STEPS - step) / max(speed, 0.01)
            print(f"Step {step:6d}/{TRAINING_STEPS} | Loss: {loss.item():.4f} | "
                  f"Avg: {avg:.4f} | {speed:.1f} it/s | ETA: {eta/60:.1f}min")

        step += 1
        if step >= TRAINING_STEPS:
            done = True
            break

total_time = time.time() - start_time
print(f"\nTraining complete! {TRAINING_STEPS} steps in {total_time/60:.1f} minutes")
print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
# Training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
window = min(200, len(losses) // 10)

axes[0].plot(losses, alpha=0.15, color='#D97757')
if window > 1:
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(losses)), smoothed, color='#D97757', linewidth=2,
                label=f'Smoothed (w={window})')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(losses, alpha=0.15, color='#D97757')
if window > 1:
    axes[1].semilogy(range(window-1, len(losses)), smoothed, color='#D97757', linewidth=2)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss (log)')
axes[1].set_title('Training Loss (log scale)', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.suptitle(f'Diffusion Policy Training: "{task_name}"', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Save checkpoint
output_dir = Path("outputs/diffusion_libero_single_task")
output_dir.mkdir(parents=True, exist_ok=True)
policy.save_pretrained(output_dir)
preprocessor.save_pretrained(output_dir)
postprocessor.save_pretrained(output_dir)
print(f"Checkpoint saved to {output_dir}")

---
## Part 4: Evaluate in the LIBERO Environment

Now the real test: does our policy actually work in the kitchen? We'll run it in the LIBERO simulator and record a video.

LIBERO tasks have a binary success criterion — the task either succeeds or fails. We'll run multiple evaluation episodes and compute the success rate.

In [ ]:
# Evaluation function for a single LIBERO task
def evaluate_single_task(policy, preprocessor, postprocessor,
                         task_idx=0, suite_name="libero_spatial",
                         n_episodes=10, max_steps=300, seed=42,
                         render_video=True):
    """Evaluate policy on a single LIBERO task. Returns success rate and optional video frames."""
    from libero.libero import benchmark
    from lerobot.envs.libero import LiberoEnv, TASK_SUITE_MAX_STEPS
    from lerobot.processor.env_processor import LiberoProcessorStep

    bench_dict = benchmark.get_benchmark_dict()
    suite = bench_dict[suite_name]()
    max_steps = TASK_SUITE_MAX_STEPS.get(suite_name, max_steps)
    libero_processor = LiberoProcessorStep()

    env = LiberoEnv(
        task_suite=suite, task_id=task_idx, task_suite_name=suite_name,
        observation_height=256, observation_width=256, obs_type="pixels_agent_pos",
    )

    task_name = suite.tasks[task_idx].language
    print(f"Evaluating: \"{task_name}\"")
    print(f"  Max steps per episode: {max_steps}")
    print(f"  Episodes: {n_episodes}\n")

    successes = []
    video_frames = []  # Collect frames from first episode for video

    policy.eval()

    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        policy.reset()
        ep_frames = []

        for step_i in range(max_steps):
            # Build observation dict
            obs_dict = {}

            # Images: extract and flip 180° to match dataset convention
            for cam_key, mapped_key in [('image', 'observation.images.image'),
                                        ('image2', 'observation.images.image2')]:
                if cam_key in obs.get('pixels', {}):
                    img = obs['pixels'][cam_key]
                    img_t = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0) / 255.0
                    img_t = torch.flip(img_t, dims=[2, 3])  # 180° flip
                    obs_dict[mapped_key] = img_t.to(device)

            # Robot state: [eef_pos(3), axis_angle(3), gripper(2)] = 8D
            if 'robot_state' in obs:
                rs = obs['robot_state']
                eef_pos = torch.tensor(rs['eef']['pos'], dtype=torch.float32)
                eef_quat = torch.tensor(rs['eef']['quat'], dtype=torch.float32)
                grip = torch.tensor(rs['gripper']['qpos'], dtype=torch.float32)
                eef_aa = libero_processor._quat2axisangle(eef_quat.unsqueeze(0)).squeeze(0)
                state = torch.cat([eef_pos, eef_aa, grip]).unsqueeze(0).to(device)
                obs_dict['observation.state'] = state

            # Preprocess, predict, postprocess
            obs_dict = preprocessor(obs_dict)
            with torch.no_grad():
                action = policy.select_action(obs_dict)
            action = postprocessor(action)
            action_np = action.squeeze(0).cpu().numpy()

            # Step environment
            obs, reward, terminated, truncated, info = env.step(action_np)

            # Collect video frame (agentview) for first episode
            if ep == 0 and render_video and 'pixels' in obs and 'image' in obs['pixels']:
                ep_frames.append(obs['pixels']['image'].copy())

            if terminated or truncated:
                break

        success = info.get('is_success', False)
        successes.append(success)
        status = 'SUCCESS' if success else 'FAIL'
        print(f"  Episode {ep+1:2d}: [{status}] ({step_i+1} steps)")

        if ep == 0:
            video_frames = ep_frames

    env.close()

    success_rate = np.mean(successes) * 100
    print(f"\n  Success rate: {success_rate:.0f}% ({sum(successes)}/{n_episodes})")

    return {
        'success_rate': success_rate,
        'successes': successes,
        'task_name': task_name,
        'video_frames': video_frames,
    }

In [ ]:
# Run evaluation!
print("=" * 60)
print("  EVALUATION: Diffusion Policy on LIBERO")
print("=" * 60)

results = evaluate_single_task(
    policy, preprocessor, postprocessor,
    task_idx=TARGET_TASK_IDX,
    suite_name="libero_spatial",
    n_episodes=10,
    render_video=True,
)

# Published baseline for reference
print(f"\nPublished LIBERO-Spatial average: 78.3% (Diffusion Policy)")
print(f"Your single-task result: {results['success_rate']:.0f}%")
if results['success_rate'] > 78.3:
    print("You beat the published multi-task baseline! (Single-task is easier though.)")

In [ ]:
# Render the evaluation video
if results['video_frames'] and len(results['video_frames']) > 0:
    frames = np.stack(results['video_frames'])
    print(f"Video: {len(frames)} frames at {meta.fps} FPS ({len(frames)/meta.fps:.1f}s)")

    # Show as inline video
    media.show_video(frames, fps=meta.fps, title=f"Eval: {results['task_name']}")

    # Also show key frames
    fig, axes = plt.subplots(1, 6, figsize=(24, 4))
    indices = np.linspace(0, len(frames)-1, 6, dtype=int)
    for i, idx in enumerate(indices):
        axes[i].imshow(frames[idx])
        t = idx / meta.fps
        axes[i].set_title(f't={t:.1f}s', fontsize=10)
        axes[i].axis('off')
    status = 'SUCCESS' if results['successes'][0] else 'FAIL'
    plt.suptitle(f'Evaluation: "{results["task_name"]}" — {status}',
                 fontsize=13, fontweight='bold',
                 color='green' if results['successes'][0] else 'red')
    plt.tight_layout()
    plt.show()
else:
    print("No video frames captured. LIBERO environment may not be available.")

---
## Part 5: Analysis — What Changed from PushT?

Let's compare the training dynamics between PushT (Week 1) and this LIBERO task.

In [ ]:
# Architecture comparison
print("=" * 65)
print("  COMPARING DIFFUSION POLICY: PushT vs LIBERO")
print("=" * 65)
print(f"{'Component':<30s} {'PushT (Week 1)':>18s} {'LIBERO (Now)':>18s}")
print("-" * 65)
print(f"{'Environment':<30s} {'2D simulation':>18s} {'3D kitchen sim':>18s}")
print(f"{'Robot':<30s} {'Point agent':>18s} {'Franka 7-DOF':>18s}")
print(f"{'Cameras':<30s} {'1 × 96×96':>18s} {'2 × 256×256':>18s}")
print(f"{'State dim':<30s} {'2':>18s} {'8':>18s}")
print(f"{'Action dim':<30s} {'2 (x, y)':>18s} {'7 (Cartesian)':>18s}")
print(f"{'crop_shape':<30s} {'(84, 84)':>18s} {f'({crop_h}, {crop_w})':>18s}")
print(f"{'Horizon':<30s} {'16':>18s} {'16':>18s}")
print(f"{'Execute':<30s} {'8':>18s} {'8':>18s}")
print(f"{'Denoising steps':<30s} {'100':>18s} {'100':>18s}")
print(f"{'Obs steps':<30s} {'2':>18s} {'2':>18s}")
print(f"{'Parameters':<30s} {'~26M':>18s} {f'~{n_params/1e6:.0f}M':>18s}")
print(f"{'Demos':<30s} {'205':>18s} {f'{len(target_episodes)}':>18s}")
print(f"{'Success metric':<30s} {'Coverage %':>18s} {'Binary success':>18s}")
print("=" * 65)
print("\nKey insight: The SAME architecture (1D U-Net + FiLM) works in both 2D")
print("and 3D. Only the input/output dimensions change. The diffusion math is")
print("identical — add noise to actions, predict the noise, denoise at inference.")

In [ ]:
# Visualize what the policy learned: overlay predictions on demo data
# Load a test sample and run the denoising process

policy.eval()
test_sample = dataset[len(dataset)//2]  # Middle of dataset

# Get the ground truth action chunk
gt_actions = test_sample['action'].numpy()  # (16, 7)

# Run prediction
obs_dict = {k: v.unsqueeze(0).to(device) if isinstance(v, torch.Tensor) else v
            for k, v in test_sample.items() if k.startswith('observation')}
obs_dict = preprocessor(obs_dict)
with torch.no_grad():
    pred_action = policy.select_action(obs_dict)
pred_action = postprocessor(pred_action)
pred_np = pred_action.squeeze(0).cpu().numpy()  # (n_action_steps, 7)

# Plot ground truth vs predicted for each action dimension
action_names = ['dx', 'dy', 'dz', 'rx', 'ry', 'rz', 'gripper']
n_pred = len(pred_np)

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for dim in range(7):
    ax = axes[dim]
    # Ground truth (full 16-step horizon)
    ax.plot(range(16), gt_actions[:, dim], 'o-', color='#5B8CB8', alpha=0.7,
            markersize=4, label='Demo (ground truth)')
    # Prediction (8 executed steps)
    ax.plot(range(n_pred), pred_np[:, dim], 's-', color='#D97757',
            markersize=5, linewidth=2, label='Policy prediction')
    ax.axvline(x=n_pred-0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(action_names[dim], fontweight='bold')
    ax.grid(alpha=0.3)
    if dim == 0:
        ax.legend(fontsize=8)

axes[7].axis('off')
axes[7].text(0.5, 0.5, f'Horizon: 16 steps\nExecute: {n_pred} steps\n\n'
             f'Blue = demo trajectory\nOrange = policy prediction\n'
             f'Dashed = execution boundary',
             ha='center', va='center', fontsize=11,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f'Ground Truth vs Predicted Actions\n"{task_name}"',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Compute MSE per dimension
mse_per_dim = ((gt_actions[:n_pred] - pred_np) ** 2).mean(axis=0)
print(f"MSE per action dimension (lower = better):")
for i, name in enumerate(action_names):
    bar = '█' * int(mse_per_dim[i] * 100)
    print(f"  {name:8s}: {mse_per_dim[i]:.4f}  {bar}")

---
## Exercises

### Exercise 1: Try a Different Task
Change `TARGET_TASK_IDX` in Part 2 to a different task (0-9 for LIBERO-Spatial). Which tasks are easier? Which are harder?

### Exercise 2: Train Longer
Increase `TRAINING_STEPS` to 50,000 or 100,000. Plot the success rate vs training steps — when does performance plateau?

### Exercise 3: Single Camera vs Dual Camera
Remove the wrist camera from `input_features` and retrain. How much does the second camera help?
```python
# Remove wrist camera
input_features_single = {
    k: ft for k, ft in input_features.items()
    if not ('image2' in k)
}
```

### Exercise 4: Compare with Week 1
Load your PushT checkpoint from Week 1. Compare:
- Training loss curves (do they converge at different rates?)
- How many steps to reach a "good" policy?
- Does the 7D action space make diffusion harder than 2D?

### Exercise 5 (Challenge): Multi-Task
Train on ALL 10 LIBERO-Spatial tasks simultaneously (use `training_episodes` from all tasks). Does the policy still succeed on your chosen task? This is the multi-task challenge from the full Week 3 notebook.

---

## Summary

| What You Did | Details |
|---|---|
| **Environment** | LIBERO kitchen — 3D manipulation with Franka Panda |
| **Task** | Single kitchen task (LIBERO-Spatial) |
| **Policy** | Diffusion Policy with 2 cameras, 8D state, 7D action |
| **Key change** | `crop_shape=(224,224)` for 256×256 images |
| **Architecture** | Same 1D U-Net + FiLM as PushT — only dimensions changed |
| **Insight** | Diffusion Policy generalizes from 2D pushing to 3D kitchen manipulation |

**Next**: The full Week 3 notebook trains on 10 tasks simultaneously and compares Diffusion vs ACT vs VQ-BeT.